<a href="https://colab.research.google.com/github/sourcesync/kagglex_gemma/blob/gw%2Finitial/colab/jorge_llamaindex_troubleshooting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install required packages

In [1]:
!pip install llama-index-llms-gemini
!pip install -q llama-index google-generativeai
!pip install llama-index-vector-stores-chroma
!pip install -q -U keras keras-nlp
!pip install -q -U keras>=3
!pip install -q -U llama_index
!pip install -q -U kagglehub --upgrade
!pip install -q llama-index-readers-web llama-index-readers-file
!pip install -q llama_index.embeddings.huggingface
!pip install -q llama-index-readers-file
!pip install -q -U weaviate-client
!pip install -q llama-index-vector-stores-weaviate
!pip install --upgrade accelerate
!pip install llama-index langfuse --upgrade

  Using cached protobuf-4.25.5-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.5-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.28.3
    Uninstalling protobuf-5.28.3:
      Successfully uninstalled protobuf-5.28.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-health-checking 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.5 which is incompatible.
grpcio-tools 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.5 which is incompatible.
tensorflow-metadata 1.16.1 requires protobuf<4.21,>=3.20.3; python_version < "3.11", but you have protobuf 4.25.5 which is incompatible.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.18.0 which is incompatible.
ERROR: pip's dependency resolver does 

# Import required packages

In [5]:
import os
from llama_index.llms.gemini import Gemini
from google.colab import files, userdata
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer
from llama_index.core import PromptTemplate
from llama_index.core.retrievers import VectorIndexAutoRetriever
from llama_index.core.vector_stores.types import MetadataInfo, VectorStoreInfo
from llama_index.core import Settings
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.schema import TextNode
from llama_index.core.settings import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.callbacks import TokenCountingHandler, CallbackManager
from langfuse.llama_index import LlamaIndexCallbackHandler

# Bind To Google Cloud
* using Colab secrets

In [3]:
google_api_token = userdata.get("google_api_key")
os.environ["GOOGLE_API_KEY"] = google_api_token

# Configure this noteboook session


In [13]:
Settings.llm = Gemini()
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
langfuse_callback_handler = LlamaIndexCallbackHandler(
  secret_key=userdata.get("langfuse_secret_key"),
  public_key="pk-lf-54cb5178-97d0-4c15-bd3f-9d549dd5adc9",
  host="https://cloud.langfuse.com"
)
Settings.callback_manager = CallbackManager([langfuse_callback_handler])

# Test Gemini
* Note you might need to enable the "Generate Language API" feature at your GCP console

In [7]:
resp = Gemini().complete("Write a poem about a magic backpack")
print(resp)

A backpack worn, a simple thing,
But holds a magic, makes it sing.
With every zip, a world unfolds,
Of wonders strange, and stories told.

A dusty book, a whispered rhyme,
Can conjure castles, in the blink of time.
A feather light, a whispered plea,
Can grant a wish, for you and me.

A compass spins, a map unfolds,
To lands unseen, where magic holds.
A tiny seed, a whispered word,
Can bloom a garden, never heard.

The backpack hums, a gentle sound,
As secrets whispered, can be found.
A hidden key, a silver thread,
Unlocks the magic, in your head.

So wear it close, this magic pack,
And let your dreams, take you back.
To worlds unknown, and stories spun,
Where anything, can be begun. 



# Create a vector dabase and retriever

In [7]:
try:
  chroma_client.delete_collection("quickstart")
except:
  pass
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("quickstart")
nodes = [
    TextNode(
        text=(
            "Michael Jordan is a retired professional basketball player,"
            " widely regarded as one of the greatest basketball players of all"
            " time."
        ),
        metadata={
            "category": "Sports",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Angelina Jolie is an American actress, filmmaker, and"
            " humanitarian. She has received numerous awards for her acting"
            " and is known for her philanthropic work."
        ),
        metadata={
            "category": "Entertainment",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Elon Musk is a business magnate, industrial designer, and"
            " engineer. He is the founder, CEO, and lead designer of SpaceX,"
            " Tesla, Inc., Neuralink, and The Boring Company."
        ),
        metadata={
            "category": "Business",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Rihanna is a Barbadian singer, actress, and businesswoman. She"
            " has achieved significant success in the music industry and is"
            " known for her versatile musical style."
        ),
        metadata={
            "category": "Music",
            "country": "Barbados",
        },
    ),
    TextNode(
        text=(
            "Cristiano Ronaldo is a Portuguese professional footballer who is"
            " considered one of the greatest football players of all time. He"
            " has won numerous awards and set multiple records during his"
            " career."
        ),
        metadata={
            "category": "Sports",
            "country": "Portugal",
        },
    ),
]
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

vector_store_info = VectorStoreInfo(
    content_info="brief biography of celebrities",
    metadata_info=[
        MetadataInfo(
            name="category",
            type="str",
            description=(
                "Category of the celebrity, one of [Sports, Entertainment,"
                " Business, Music]"
            ),
        ),
        MetadataInfo(
            name="country",
            type="str",
            description=(
                "Country of the celebrity, one of [United States, Barbados,"
                " Portugal]"
            ),
        ),
    ],
)
retriever = VectorIndexAutoRetriever(
    index, vector_store_info=vector_store_info
)

# Test the retriever

In [8]:
retriever.retrieve("Who is a famous footballer?")

[NodeWithScore(node=TextNode(id_='b23311d9-514b-4fbe-89b3-b32504b30961', embedding=None, metadata={'category': 'Sports', 'country': 'Portugal'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='Cristiano Ronaldo is a Portuguese professional footballer who is considered one of the greatest football players of all time. He has won numerous awards and set multiple records during his career.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.6436362223188633),
 NodeWithScore(node=TextNode(id_='b80d2895-ca06-4119-a0f8-85843c01f609', embedding=None, metadata={'category': 'Sports', 'country': 'United States'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='Michael Jordan is a retired professional basketball player, widely regarded as one of the greatest basketball players of all time.', 

# Define the prompt template

In [9]:
# Write prompt template with functions
qa_prompt_tmpl_str = """\
Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, \
answer the query asking about Machine Learning Concepts to learn about and provide potential courses\
to the user from the Context information above.\
Below is a example template follow the formatting in the answer when responding to a user's query

Query: I want to learn about llm and how to finetune them. I'm intermediate and i want to build a rag pipeline'?
Answer:
| Course/Module | Source | Level | Duration (Estimate) | Keywords | Reason |Link|
|---------------------------------------------------|---------------------------------------|-------------|----------------------|---------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------|------ |
| **Introduction to Large Language Models (LLMs)** | Various online courses (e.g., Coursera, edX) | Beginner/Intermediate | Varies (4-8 weeks) | LLMs, transformers, attention mechanisms, language modeling, tokenization | Provides foundational knowledge of LLMs, their architecture, and capabilities. Essential before tackling fine-tuning or RAG. | ** Link to Course ** |
| **Natural Language Processing (NLP) Fundamentals** | Various online courses (e.g., Stanford NLP) | Intermediate | Varies (6-10 weeks) | NLP, text preprocessing, word embeddings, sentiment analysis, named entity recognition | Necessary for understanding how LLMs process and understand text. Many RAG techniques rely on NLP for data preprocessing and query understanding. | ** Link to Course ** |
| **Fine-tuning LLMs** | Hugging Face Course, Papers with Code | Intermediate/Advanced | Varies (2-4 weeks) | Fine-tuning, transfer learning, hyperparameter tuning, model evaluation | Teaches you how to adapt pre-trained LLMs to specific tasks, crucial for building a high-performing RAG system. Learn techniques like prompt engineering. | ** Link to Course ** |
| **Retrieval Augmented Generation (RAG) Techniques** | Research Papers, Blogs, Tutorials | Advanced | Varies (Ongoing Study) | RAG, vector databases, embedding generation, knowledge retrieval, question answering | Focuses on the architecture and implementation of RAG pipelines. You\'ll learn to select and integrate components like vector databases (e.g., Pinecone, Weaviate) and retrieval methods. | ** Link to Course ** |
| **Python for Data Science (if needed)** | DataCamp, Codecademy, Fast.ai | Intermediate | Varies (2-4 weeks) | Python, pandas, numpy, scikit-learn | Reinforce your Python skills for data manipulation and model building within your RAG pipeline. | ** Link to Course ** |
| **Vector Databases (if needed)** | Pinecone, Weaviate documentation | Intermediate | Varies (1-2 weeks) | Vector databases, similarity search, indexing, scalability | Understanding vector databases is crucial for efficient knowledge retrieval in a RAG pipeline. Learn about different databases and their strengths. | ** Link to Course **|



Query: {query_str}
Answer: \
"""


# Compose the RAG-based query engine

In [12]:
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

response_synthesizer = get_response_synthesizer(text_qa_template = qa_prompt_tmpl)

advanced_rag_query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)
response = advanced_rag_query_engine.query(' I want to learn about RAG')

print(str(response))

Answer: 
| Course/Module | Source | Level | Duration (Estimate) | Keywords | Reason |Link|
|---------------------------------------------------|---------------------------------------|-------------|----------------------|---------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------|------ |
| **Introduction to Large Language Models (LLMs)** | Various online courses (e.g., Coursera, edX) | Beginner/Intermediate | Varies (4-8 weeks) | LLMs, transformers, attention mechanisms, language modeling, tokenization | Provides foundational knowledge of LLMs, their architecture, and capabilities. Essential before tackling fine-tuning or RAG. | ** Link to Course ** |
| **Natural Language Processing (NLP) Fundamentals** | Various online courses (e.g., Stanford NLP) | Intermediate | Varies (6-10 weeks) | NLP, text preproces